# 02 - LV traits & pathways tables

Loads SHAP results from `01_LV_importance_kmeans` and produces clean trait / pathway dataframes for every threshold combination.

💡 **Environment:** `clamp-analyses`

In [73]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pyprojroot.here import here
import urllib.request

import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri
readRDS = ro.r['readRDS']


# Settings

In [74]:
OUTPUT_DIR   = here('output/gtex_feature_importance_kmeans_binary_shap')
MIN_PURITY   = 0.70
MIN_ACCURACY = 0.95
MIN_N        = 10_000   # min GWAS sample size for traits

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Load data

In [75]:
accuracy_df     = pd.read_csv(OUTPUT_DIR / 'accuracy_summary.tsv',     sep='\t')
shap_results_df = pd.read_csv(OUTPUT_DIR / 'all_shap_positive.tsv',    sep='\t')
cumulative_df   = pd.read_csv(OUTPUT_DIR / 'cumulative_importance.tsv', sep='\t')

print(accuracy_df.shape, shap_results_df.shape, cumulative_df.shape)


(36, 10) (18024, 6) (18024, 5)


In [76]:
phenoplier = pd.read_csv(
    here('data/gtex/phenoplier/gtex-gls-summary-phenomexcan.tsv.gz'),
    sep='\t', compression='gzip', low_memory=False
)[['lv', 'phenotype_desc', 'fdr']].rename(
    columns={'lv': 'LV', 'phenotype_desc': 'phenotype'})
print(phenoplier.shape)


(1668188, 3)


In [77]:
pheno_info_path = here('output/gtex/phenomexcan_simplified_phenotypes_info.tsv.gz')
if not pheno_info_path.exists():
    url = 'https://zenodo.org/records/14941353/files/phenomexcan-phenotypes_info.tsv.gz?download=1'
    urllib.request.urlretrieve(url, pheno_info_path)

pheno_info = pd.read_csv(pheno_info_path, sep='\t', compression='gzip', index_col='pheno_id')
pheno_info['n_eff'] = pheno_info['n'].combine_first(
    pheno_info['n_cases'] + pheno_info['n_controls'])
desc_to_n = (
    pheno_info.dropna(subset=['n_eff'])
    .groupby('description')['n_eff'].max().astype(int).to_dict()
)
print(f'Phenotypes with n: {len(desc_to_n)}')


Phenotypes with n: 4034


In [78]:
rds = readRDS(str(here('output/gtex/CLAMPfull.rds')))
sm  = rds.rx2('summary')
with localconverter(ro.default_converter + pandas2ri.converter):
    sm_vals = ro.conversion.rpy2py(sm)
summary_df = pd.DataFrame(
    data=sm_vals,
    index=sm.rownames if sm.rownames else None,
    columns=sm.colnames if sm.colnames else None,
)
summary_df['pathway'] = summary_df['pathway'].str.replace('C2CP_', '', regex=False)
print(summary_df.shape)


(2377, 7)


# Valid tissues

In [79]:
valid_tissues = (
    accuracy_df[
        (accuracy_df['Mean_Purity']                >= MIN_PURITY) &
        (accuracy_df['Best_Test_Balanced_Accuracy'] >= MIN_ACCURACY)
    ]['Tissue'].sort_values().tolist()
)
print(f'{len(valid_tissues)} valid tissues')
accuracy_df[accuracy_df['Tissue'].isin(valid_tissues)][
    ['Tissue','Mean_Purity','Best_Test_Balanced_Accuracy','LVs_for_80pct','LVs_for_90pct']
].set_index('Tissue').sort_values('Best_Test_Balanced_Accuracy', ascending=False)


25 valid tissues


,Mean_Purity,Best_Test_Balanced_Accuracy,LVs_for_80pct,LVs_for_90pct
Tissue,,,,
Adrenal Gland,1.000000,1.000000,34,63
Esophagus - Mucosa,0.858696,1.000000,40,86
Artery - Aorta,1.000000,1.000000,42,75
Cells - EBV-transformed lymphocytes,1.000000,1.000000,41,65
Cells - Cultured fibroblasts,1.000000,1.000000,40,71
Minor Salivary Gland,0.986395,1.000000,35,70
Lung,0.994280,1.000000,29,57
Liver,0.755853,1.000000,70,128
Heart - Atrial Appendage,0.995204,1.000000,48,93


# Helper functions

In [80]:
def top_lvs_at_pct(pct):
    """Return {tissue: [lv, ...]} covering `pct`% cumulative SHAP."""
    out = {}
    for tissue in valid_tissues:
        df_t  = cumulative_df[cumulative_df['Tissue'] == tissue].sort_values('Rank')
        mask  = df_t['Cumulative_Percent'] >= pct
        n     = int(df_t.loc[mask, 'Rank'].iloc[0]) if mask.any() else len(df_t)
        out[tissue] = shap_results_df[
            shap_results_df['Tissue'] == tissue
        ]['Feature'].head(n).tolist()
    return out


def trait_table(top_per_tissue, fdr_thresh, use_seen_lvs=True):
    """
    Build traits dataframe.
    Columns: Tissue | LV | shap_rank | phenotype | FDR | n_gwas
    """
    phen = phenoplier[phenoplier['fdr'] < fdr_thresh].copy()
    phen['n_gwas'] = phen['phenotype'].map(desc_to_n)
    phen = phen[phen['n_gwas'] >= MIN_N]

    rows, seen = [], set()
    for tissue in valid_tissues:
        for rank, lv in enumerate(top_per_tissue.get(tissue, []), 1):
            if use_seen_lvs and lv in seen:
                continue
            hits = phen[phen['LV'] == lv].sort_values('fdr')
            for _, r in hits.iterrows():
                rows.append(dict(Tissue=tissue, LV=lv, shap_rank=rank,
                                 phenotype=r['phenotype'],
                                 FDR=r['fdr'], n_gwas=int(r['n_gwas'])))
            if len(hits):
                seen.add(lv)
    return pd.DataFrame(rows)


def pathway_table(top_per_tissue, fdr_thresh, auc_thresh, use_seen_lvs=True):
    """
    Build pathways dataframe.
    Columns: Tissue | LV | shap_rank | pathway | AUC | FDR
    """
    summ = summary_df[
        (summary_df['AUC'] > auc_thresh) &
        (summary_df['FDR'] < fdr_thresh)
    ]
    rows, seen = [], set()
    for tissue in valid_tissues:
        for rank, lv in enumerate(top_per_tissue.get(tissue, []), 1):
            if use_seen_lvs and lv in seen:
                continue
            hits = summ[summ['LV'] == lv].sort_values('FDR')
            for _, r in hits.iterrows():
                rows.append(dict(Tissue=tissue, LV=lv, shap_rank=rank,
                                 pathway=r['pathway'],
                                 AUC=round(float(r['AUC']), 4),
                                 FDR=r['FDR']))
            if len(hits):
                seen.add(lv)
    return pd.DataFrame(rows)


# LV count comparison across cumulative thresholds
lv_counts = pd.DataFrame(
    {f'LVs@{p}%': [len(top_lvs_at_pct(p).get(t, [])) for t in valid_tissues]
     for p in [30, 40, 50, 60, 70]},
    index=valid_tissues
)
lv_counts.index.name = 'Tissue'
display(lv_counts)


,LVs@30%,LVs@40%,LVs@50%,LVs@60%,LVs@70%
Tissue,,,,,
Adipose - Subcutaneous,6,9,15,24,39
Adipose - Visceral (Omentum),7,12,19,30,46
Adrenal Gland,3,4,6,11,20
Artery - Aorta,5,8,12,17,27
Artery - Coronary,6,10,14,22,32
Artery - Tibial,5,8,13,19,28
Cells - Cultured fibroblasts,6,8,12,18,26
Cells - EBV-transformed lymphocytes,5,9,14,20,29
Esophagus - Mucosa,3,4,7,12,22


# Traits

Each cell below is one combination: `(cumulative %, FDR threshold, seen_lvs)`.

In [81]:
# pct=40%  FDR<0.01  seen_lvs=True
_tpt = top_lvs_at_pct(40)
_df  = trait_table(_tpt, fdr_thresh=0.01, use_seen_lvs=True)
print(f'pct=40%  FDR<0.01  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=40%  FDR<0.01  seen_lvs=True  ->  344 rows, 23 tissues, 52 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
1,Adipose - Subcutaneous,LV378,Facial ageing
2,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
3,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
4,Adipose - Visceral (Omentum),LV376,Heel broadband ultrasound attenuation (right)
5,Adipose - Visceral (Omentum),LV376,Heel broadband ultrasound attenuation (left)
6,Adipose - Visceral (Omentum),LV376,"Heel bone mineral density (BMD) T-score, automated (right)"
7,Adipose - Visceral (Omentum),LV376,"Heel quantitative ultrasound index (QUI), direct entry (right)"
8,Adipose - Visceral (Omentum),LV376,Heel bone mineral density (BMD) (right)
9,Adipose - Visceral (Omentum),LV376,Heel bone mineral density (BMD) (left)


In [82]:
# pct=40%  FDR<0.01  seen_lvs=False
_tpt = top_lvs_at_pct(40)
_df  = trait_table(_tpt, fdr_thresh=0.01, use_seen_lvs=False)
print(f'pct=40%  FDR<0.01  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=40%  FDR<0.01  seen_lvs=False  ->  535 rows, 25 tissues, 52 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
1,Adipose - Subcutaneous,LV378,Facial ageing
2,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
3,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
4,Adipose - Visceral (Omentum),LV376,Heel broadband ultrasound attenuation (right)
5,Adipose - Visceral (Omentum),LV376,Heel broadband ultrasound attenuation (left)
6,Adipose - Visceral (Omentum),LV376,"Heel bone mineral density (BMD) T-score, automated (right)"
7,Adipose - Visceral (Omentum),LV376,"Heel quantitative ultrasound index (QUI), direct entry (right)"
8,Adipose - Visceral (Omentum),LV376,Heel bone mineral density (BMD) (right)
9,Adipose - Visceral (Omentum),LV376,Heel bone mineral density (BMD) (left)


In [83]:
# pct=40%  FDR<0.05  seen_lvs=True
_tpt = top_lvs_at_pct(40)
_df  = trait_table(_tpt, fdr_thresh=0.05, use_seen_lvs=True)
print(f'pct=40%  FDR<0.05  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=40%  FDR<0.05  seen_lvs=True  ->  515 rows, 24 tissues, 80 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV263,Sum of road length of major roads within 100m
1,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
2,Adipose - Subcutaneous,LV378,Facial ageing
3,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
4,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
5,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Light brown"
6,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Black"
7,Adipose - Subcutaneous,LV90,"Non-cancer illness code, self-reported: polycythaemia vera"
8,Adipose - Subcutaneous,LV230,Type of fat/oil used in cooking: Vegetable oil
9,Adipose - Visceral (Omentum),LV376,Heel broadband ultrasound attenuation (right)


In [84]:
# pct=40%  FDR<0.05  seen_lvs=False
_tpt = top_lvs_at_pct(40)
_df  = trait_table(_tpt, fdr_thresh=0.05, use_seen_lvs=False)
print(f'pct=40%  FDR<0.05  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=40%  FDR<0.05  seen_lvs=False  ->  789 rows, 25 tissues, 80 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV263,Sum of road length of major roads within 100m
1,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
2,Adipose - Subcutaneous,LV378,Facial ageing
3,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
4,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
5,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Light brown"
6,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Black"
7,Adipose - Subcutaneous,LV90,"Non-cancer illness code, self-reported: polycythaemia vera"
8,Adipose - Subcutaneous,LV230,Type of fat/oil used in cooking: Vegetable oil
9,Adipose - Visceral (Omentum),LV263,Sum of road length of major roads within 100m


In [85]:
# pct=50%  FDR<0.01  seen_lvs=True
_tpt = top_lvs_at_pct(50)
_df  = trait_table(_tpt, fdr_thresh=0.01, use_seen_lvs=True)
print(f'pct=50%  FDR<0.01  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=50%  FDR<0.01  seen_lvs=True  ->  461 rows, 22 tissues, 74 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
1,Adipose - Subcutaneous,LV378,Facial ageing
2,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
3,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
4,Adipose - Subcutaneous,LV145,"Hair colour (natural, before greying): Light brown"
5,Adipose - Subcutaneous,LV111,"Non-cancer illness code, self-reported: rheumatoid arthritis"
6,Adipose - Subcutaneous,LV111,Rheumatoid arthritis
7,Adipose - Subcutaneous,LV111,Other/unspecified seropositiverheumatoid arthritis
8,Adipose - Subcutaneous,LV111,Diagnoses - main ICD10: M06 Other rheumatoid arthritis
9,Adipose - Subcutaneous,LV111,Other/unspecified rheumatoid arthritis


In [86]:
# pct=50%  FDR<0.01  seen_lvs=False
_tpt = top_lvs_at_pct(50)
_df  = trait_table(_tpt, fdr_thresh=0.01, use_seen_lvs=False)
print(f'pct=50%  FDR<0.01  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=50%  FDR<0.01  seen_lvs=False  ->  875 rows, 25 tissues, 74 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
1,Adipose - Subcutaneous,LV378,Facial ageing
2,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
3,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
4,Adipose - Subcutaneous,LV145,"Hair colour (natural, before greying): Light brown"
5,Adipose - Subcutaneous,LV111,"Non-cancer illness code, self-reported: rheumatoid arthritis"
6,Adipose - Subcutaneous,LV111,Rheumatoid arthritis
7,Adipose - Subcutaneous,LV111,Other/unspecified seropositiverheumatoid arthritis
8,Adipose - Subcutaneous,LV111,Diagnoses - main ICD10: M06 Other rheumatoid arthritis
9,Adipose - Subcutaneous,LV111,Other/unspecified rheumatoid arthritis


In [87]:
# pct=50%  FDR<0.05  seen_lvs=True
_tpt = top_lvs_at_pct(50)
_df  = trait_table(_tpt, fdr_thresh=0.05, use_seen_lvs=True)
print(f'pct=50%  FDR<0.05  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=50%  FDR<0.05  seen_lvs=True  ->  686 rows, 24 tissues, 111 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV263,Sum of road length of major roads within 100m
1,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
2,Adipose - Subcutaneous,LV378,Facial ageing
3,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
4,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
5,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Light brown"
6,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Black"
7,Adipose - Subcutaneous,LV90,"Non-cancer illness code, self-reported: polycythaemia vera"
8,Adipose - Subcutaneous,LV230,Type of fat/oil used in cooking: Vegetable oil
9,Adipose - Subcutaneous,LV145,"Hair colour (natural, before greying): Light brown"


In [88]:
# pct=50%  FDR<0.05  seen_lvs=False
_tpt = top_lvs_at_pct(50)
_df  = trait_table(_tpt, fdr_thresh=0.05, use_seen_lvs=False)
print(f'pct=50%  FDR<0.05  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'phenotype']]

pct=50%  FDR<0.05  seen_lvs=False  ->  1287 rows, 25 tissues, 111 LVs


,Tissue,LV,phenotype
0,Adipose - Subcutaneous,LV263,Sum of road length of major roads within 100m
1,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: Hayfever, allergic rhinitis or eczema"
2,Adipose - Subcutaneous,LV378,Facial ageing
3,Adipose - Subcutaneous,LV378,"Age hay fever, rhinitis or eczema diagnosed"
4,Adipose - Subcutaneous,LV378,"Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor: None of the above"
5,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Light brown"
6,Adipose - Subcutaneous,LV378,"Hair colour (natural, before greying): Black"
7,Adipose - Subcutaneous,LV90,"Non-cancer illness code, self-reported: polycythaemia vera"
8,Adipose - Subcutaneous,LV230,Type of fat/oil used in cooking: Vegetable oil
9,Adipose - Subcutaneous,LV145,"Hair colour (natural, before greying): Light brown"


# Pathways

Each cell: `(cumulative %, AUC threshold, FDR threshold, seen_lvs)`.

In [89]:
# pct=40%  FDR<0.05  AUC>0.7  seen_lvs=True
_tpt = top_lvs_at_pct(40)
_df  = pathway_table(_tpt, fdr_thresh=0.05, auc_thresh=0.7, use_seen_lvs=True)
print(f'pct=40%  FDR<0.05  AUC>0.7  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=40%  FDR<0.05  AUC>0.7  seen_lvs=True  ->  37 rows, 13 tissues, 17 LVs


,Tissue,LV,pathway
0,Adipose - Visceral (Omentum),LV57,BP_Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)
1,Adipose - Visceral (Omentum),LV57,BP_Aerobic Electron Transport Chain (GO:0019646)
2,Adipose - Visceral (Omentum),LV57,BP_Aerobic Respiration (GO:0009060)
3,Adrenal Gland,LV3,BP_Negative Regulation of T Cell Mediated Immunity (GO:0002710)
4,Artery - Aorta,LV145,BP_Skeletal Muscle Contraction (GO:0003009)
5,Artery - Coronary,LV99,BP_Dopaminergic Neuron Differentiation (GO:0071542)
6,Artery - Tibial,LV81,BP_Epidermis Development (GO:0008544)
7,Cells - EBV-transformed lymphocytes,LV123,BP_Reverse Cholesterol Transport (GO:0043691)
8,Cells - EBV-transformed lymphocytes,LV116,BP_Cilium Movement (GO:0003341)
9,Cells - EBV-transformed lymphocytes,LV244,BP_Mitochondrial Gene Expression (GO:0140053)


In [90]:
# pct=40%  FDR<0.05  AUC>0.7  seen_lvs=False
_tpt = top_lvs_at_pct(40)
_df  = pathway_table(_tpt, fdr_thresh=0.05, auc_thresh=0.7, use_seen_lvs=False)
print(f'pct=40%  FDR<0.05  AUC>0.7  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=40%  FDR<0.05  AUC>0.7  seen_lvs=False  ->  48 rows, 14 tissues, 17 LVs


,Tissue,LV,pathway
0,Adipose - Visceral (Omentum),LV57,BP_Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)
1,Adipose - Visceral (Omentum),LV57,BP_Aerobic Electron Transport Chain (GO:0019646)
2,Adipose - Visceral (Omentum),LV57,BP_Aerobic Respiration (GO:0009060)
3,Adrenal Gland,LV3,BP_Negative Regulation of T Cell Mediated Immunity (GO:0002710)
4,Artery - Aorta,LV145,BP_Skeletal Muscle Contraction (GO:0003009)
5,Artery - Coronary,LV99,BP_Dopaminergic Neuron Differentiation (GO:0071542)
6,Artery - Tibial,LV81,BP_Epidermis Development (GO:0008544)
7,Cells - EBV-transformed lymphocytes,LV123,BP_Reverse Cholesterol Transport (GO:0043691)
8,Cells - EBV-transformed lymphocytes,LV116,BP_Cilium Movement (GO:0003341)
9,Cells - EBV-transformed lymphocytes,LV244,BP_Mitochondrial Gene Expression (GO:0140053)


In [91]:
# pct=40%  FDR<0.1  AUC>0.6  seen_lvs=True
_tpt = top_lvs_at_pct(40)
_df  = pathway_table(_tpt, fdr_thresh=0.1, auc_thresh=0.6, use_seen_lvs=True)
print(f'pct=40%  FDR<0.1  AUC>0.6  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=40%  FDR<0.1  AUC>0.6  seen_lvs=True  ->  100 rows, 19 tissues, 36 LVs


,Tissue,LV,pathway
0,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Lipid Catabolic Process (GO:0050996)
1,Adipose - Subcutaneous,LV263,BP_Regulation of Fatty Acid Beta-Oxidation (GO:0031998)
2,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Fatty Acid Oxidation (GO:0046321)
3,Adipose - Visceral (Omentum),LV83,BP_Skeletal Muscle Contraction (GO:0003009)
4,Adipose - Visceral (Omentum),LV83,BP_Actin-Myosin Filament Sliding (GO:0033275)
5,Adipose - Visceral (Omentum),LV57,BP_Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)
6,Adipose - Visceral (Omentum),LV57,BP_Aerobic Electron Transport Chain (GO:0019646)
7,Adipose - Visceral (Omentum),LV57,BP_Aerobic Respiration (GO:0009060)
8,Adipose - Visceral (Omentum),LV57,"BP_Mitochondrial Electron Transport, Ubiquinol to Cytochrome C (GO:0006122)"
9,Adipose - Visceral (Omentum),LV106,BP_Endosome Transport via Multivesicular Body Sorting Pathway (GO:0032509)


In [92]:
# pct=40%  FDR<0.1  AUC>0.6  seen_lvs=False
_tpt = top_lvs_at_pct(40)
_df  = pathway_table(_tpt, fdr_thresh=0.1, auc_thresh=0.6, use_seen_lvs=False)
print(f'pct=40%  FDR<0.1  AUC>0.6  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=40%  FDR<0.1  AUC>0.6  seen_lvs=False  ->  133 rows, 23 tissues, 36 LVs


,Tissue,LV,pathway
0,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Lipid Catabolic Process (GO:0050996)
1,Adipose - Subcutaneous,LV263,BP_Regulation of Fatty Acid Beta-Oxidation (GO:0031998)
2,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Fatty Acid Oxidation (GO:0046321)
3,Adipose - Visceral (Omentum),LV263,BP_Positive Regulation of Lipid Catabolic Process (GO:0050996)
4,Adipose - Visceral (Omentum),LV263,BP_Regulation of Fatty Acid Beta-Oxidation (GO:0031998)
5,Adipose - Visceral (Omentum),LV263,BP_Positive Regulation of Fatty Acid Oxidation (GO:0046321)
6,Adipose - Visceral (Omentum),LV83,BP_Skeletal Muscle Contraction (GO:0003009)
7,Adipose - Visceral (Omentum),LV83,BP_Actin-Myosin Filament Sliding (GO:0033275)
8,Adipose - Visceral (Omentum),LV57,BP_Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)
9,Adipose - Visceral (Omentum),LV57,BP_Aerobic Electron Transport Chain (GO:0019646)


In [93]:
# pct=50%  FDR<0.05  AUC>0.7  seen_lvs=True
_tpt = top_lvs_at_pct(50)
_df  = pathway_table(_tpt, fdr_thresh=0.05, auc_thresh=0.7, use_seen_lvs=True)
print(f'pct=50%  FDR<0.05  AUC>0.7  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=50%  FDR<0.05  AUC>0.7  seen_lvs=True  ->  47 rows, 14 tissues, 21 LVs


,Tissue,LV,pathway
0,Adipose - Subcutaneous,LV145,BP_Skeletal Muscle Contraction (GO:0003009)
1,Adipose - Visceral (Omentum),LV57,BP_Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)
2,Adipose - Visceral (Omentum),LV57,BP_Aerobic Electron Transport Chain (GO:0019646)
3,Adipose - Visceral (Omentum),LV57,BP_Aerobic Respiration (GO:0009060)
4,Adipose - Visceral (Omentum),LV2,BP_Regulation of Calcium Ion-Dependent Exocytosis (GO:0017158)
5,Adrenal Gland,LV3,BP_Negative Regulation of T Cell Mediated Immunity (GO:0002710)
6,Artery - Aorta,LV81,BP_Epidermis Development (GO:0008544)
7,Artery - Coronary,LV99,BP_Dopaminergic Neuron Differentiation (GO:0071542)
8,Cells - EBV-transformed lymphocytes,LV123,BP_Reverse Cholesterol Transport (GO:0043691)
9,Cells - EBV-transformed lymphocytes,LV116,BP_Cilium Movement (GO:0003341)


In [94]:
# pct=50%  FDR<0.05  AUC>0.7  seen_lvs=False
_tpt = top_lvs_at_pct(50)
_df  = pathway_table(_tpt, fdr_thresh=0.05, auc_thresh=0.7, use_seen_lvs=False)
print(f'pct=50%  FDR<0.05  AUC>0.7  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=50%  FDR<0.05  AUC>0.7  seen_lvs=False  ->  86 rows, 21 tissues, 21 LVs


,Tissue,LV,pathway
0,Adipose - Subcutaneous,LV145,BP_Skeletal Muscle Contraction (GO:0003009)
1,Adipose - Visceral (Omentum),LV57,BP_Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)
2,Adipose - Visceral (Omentum),LV57,BP_Aerobic Electron Transport Chain (GO:0019646)
3,Adipose - Visceral (Omentum),LV57,BP_Aerobic Respiration (GO:0009060)
4,Adipose - Visceral (Omentum),LV2,BP_Regulation of Calcium Ion-Dependent Exocytosis (GO:0017158)
5,Adrenal Gland,LV3,BP_Negative Regulation of T Cell Mediated Immunity (GO:0002710)
6,Artery - Aorta,LV145,BP_Skeletal Muscle Contraction (GO:0003009)
7,Artery - Aorta,LV81,BP_Epidermis Development (GO:0008544)
8,Artery - Coronary,LV99,BP_Dopaminergic Neuron Differentiation (GO:0071542)
9,Artery - Tibial,LV81,BP_Epidermis Development (GO:0008544)


In [95]:
# pct=50%  FDR<0.1  AUC>0.6  seen_lvs=True
_tpt = top_lvs_at_pct(50)
_df  = pathway_table(_tpt, fdr_thresh=0.1, auc_thresh=0.6, use_seen_lvs=True)
print(f'pct=50%  FDR<0.1  AUC>0.6  seen_lvs=True  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=50%  FDR<0.1  AUC>0.6  seen_lvs=True  ->  134 rows, 22 tissues, 47 LVs


,Tissue,LV,pathway
0,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Lipid Catabolic Process (GO:0050996)
1,Adipose - Subcutaneous,LV263,BP_Regulation of Fatty Acid Beta-Oxidation (GO:0031998)
2,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Fatty Acid Oxidation (GO:0046321)
3,Adipose - Subcutaneous,LV145,BP_Skeletal Muscle Contraction (GO:0003009)
4,Adipose - Subcutaneous,LV145,BP_Embryonic Skeletal System Morphogenesis (GO:0048704)
5,Adipose - Subcutaneous,LV111,BP_Antigen Processing and Presentation of Endogenous Peptide Antigen (GO:0002483)
6,Adipose - Subcutaneous,LV111,BP_Ag Processing and Presentation of Endogenous Pep Ag via MHC Cls I via ER Pway (GO:0002484)
7,Adipose - Subcutaneous,LV111,BP_Antigen Processing and Presentation of Endogenous Peptide Antigen via MHC Class Ib (GO:0002476)
8,Adipose - Subcutaneous,LV111,"BP_Ag Processing and Presentation of Endogenous Pep Ag via MHC Cls I via ER Pway, TAP-ind (GO:0002486)"
9,Adipose - Subcutaneous,LV93,BP_Positive Regulation of Ruffle Assembly (GO:1900029)


In [96]:
# pct=50%  FDR<0.1  AUC>0.6  seen_lvs=False
_tpt = top_lvs_at_pct(50)
_df  = pathway_table(_tpt, fdr_thresh=0.1, auc_thresh=0.6, use_seen_lvs=False)
print(f'pct=50%  FDR<0.1  AUC>0.6  seen_lvs=False  ->  {len(_df)} rows, {_df["Tissue"].nunique()} tissues, {_df["LV"].nunique()} LVs')
_df[['Tissue', 'LV', 'pathway']]

pct=50%  FDR<0.1  AUC>0.6  seen_lvs=False  ->  256 rows, 25 tissues, 47 LVs


,Tissue,LV,pathway
0,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Lipid Catabolic Process (GO:0050996)
1,Adipose - Subcutaneous,LV263,BP_Regulation of Fatty Acid Beta-Oxidation (GO:0031998)
2,Adipose - Subcutaneous,LV263,BP_Positive Regulation of Fatty Acid Oxidation (GO:0046321)
3,Adipose - Subcutaneous,LV145,BP_Skeletal Muscle Contraction (GO:0003009)
4,Adipose - Subcutaneous,LV145,BP_Embryonic Skeletal System Morphogenesis (GO:0048704)
5,Adipose - Subcutaneous,LV111,BP_Antigen Processing and Presentation of Endogenous Peptide Antigen (GO:0002483)
6,Adipose - Subcutaneous,LV111,BP_Ag Processing and Presentation of Endogenous Pep Ag via MHC Cls I via ER Pway (GO:0002484)
7,Adipose - Subcutaneous,LV111,BP_Antigen Processing and Presentation of Endogenous Peptide Antigen via MHC Class Ib (GO:0002476)
8,Adipose - Subcutaneous,LV111,"BP_Ag Processing and Presentation of Endogenous Pep Ag via MHC Cls I via ER Pway, TAP-ind (GO:0002486)"
9,Adipose - Subcutaneous,LV93,BP_Positive Regulation of Ruffle Assembly (GO:1900029)
